In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import pandas as pd
from sklearn.model_selection import train_test_split
import time
import copy

# Vérification de la disponibilité GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda:0


In [4]:
# Exploration de la structure des dossiers
def explore_dataset_structure(base_path="compCars/CompCars/data"):
    print("Structure du dataset:")
    print("=" * 50)
    
    # Compter le nombre de marques, modèles et images
    make_count = 0
    model_count = 0
    image_count = 0
    years = set()
    
    image_path = os.path.join(base_path, "image")
    if os.path.exists(image_path):
        for make_id in os.listdir(image_path):
            make_path = os.path.join(image_path, make_id)
            if os.path.isdir(make_path):
                make_count += 1
                for model_id in os.listdir(make_path):
                    model_path = os.path.join(make_path, model_id)
                    if os.path.isdir(model_path):
                        model_count += 1
                        for year in os.listdir(model_path):
                            year_path = os.path.join(model_path, year)
                            if os.path.isdir(year_path):
                                years.add(year)
                                for img_file in os.listdir(year_path):
                                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                        image_count += 1
    
    print(f"Nombre de marques: {make_count}")
    print(f"Nombre de modèles: {model_count}")
    print(f"Nombre d'années différentes: {len(years)}")
    print(f"Nombre total d'images: {image_count}")
    
    return make_count, model_count, image_count

# Exécuter l'exploration
make_count, model_count, total_images = explore_dataset_structure()

Structure du dataset:
Nombre de marques: 163
Nombre de modèles: 1716
Nombre d'années différentes: 27
Nombre total d'images: 136726


In [5]:
# Charger les noms des marques et modèles
def load_make_model_names(mat_path="compCars/CompCars/data/misc/make_model_name.mat"):
    mat_data = loadmat(mat_path)
    
    # Les noms sont généralement stockés dans des arrays de cellules MATLAB
    make_names = []
    model_names = []
    
    if 'make_names' in mat_data:
        make_names = [str(name[0]) for name in mat_data['make_names'].flatten()]
    
    if 'model_names' in mat_data:
        model_names = [str(name[0]) for name in mat_data['model_names'].flatten()]
    
    print(f"Nombre de noms de marques chargés: {len(make_names)}")
    print(f"Nombre de noms de modèles chargés: {len(model_names)}")
    
    return make_names, model_names

# Charger les attributs
def load_attributes(attr_path="data/misc/attributes.txt"):
    attributes = {}
    if os.path.exists(attr_path):
        with open(attr_path, 'r') as f:
            for line in f:
                if ':' in line:
                    key, value = line.strip().split(':', 1)
                    attributes[key.strip()] = value.strip()
    return attributes

# Charger les données
make_names, model_names = load_make_model_names()
attributes = load_attributes()

print("Quelques marques:", make_names[:10])
print("Quelques attributs:", list(attributes.keys())[:5])

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
class CarMakeDataset(Dataset):
    def __init__(self, base_path="data/image", transform=None, make_names=None):
        self.base_path = base_path
        self.transform = transform
        self.make_names = make_names
        self.data = []
        self.make_to_idx = {}
        self.idx_to_make = {}
        
        # Parcourir la structure et créer le mapping
        if os.path.exists(base_path):
            make_ids = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])
            
            # Créer le mapping make_id -> index
            for idx, make_id in enumerate(make_ids):
                self.make_to_idx[make_id] = idx
                self.idx_to_make[idx] = make_id
            
            # Collecter toutes les images avec leurs labels
            for make_id in make_ids:
                make_path = os.path.join(base_path, make_id)
                make_idx = self.make_to_idx[make_id]
                
                for model_id in os.listdir(make_path):
                    model_path = os.path.join(make_path, model_id)
                    if os.path.isdir(model_path):
                        for year in os.listdir(model_path):
                            year_path = os.path.join(model_path, year)
                            if os.path.isdir(year_path):
                                for img_file in os.listdir(year_path):
                                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                        img_path = os.path.join(year_path, img_file)
                                        self.data.append((img_path, make_idx))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, make_idx = self.data[idx]
        
        # Charger l'image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, make_idx

# Transformations pour l'entraînement
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Créer le dataset complet
full_dataset = CarMakeDataset(transform=train_transform, make_names=make_names)

# Split train/val
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)

# Appliquer différentes transformations pour la validation
val_dataset.dataset.transform = val_transform

# Créer les DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes (makes): {len(full_dataset.make_to_idx)}")

In [ ]:
class CarMakeModel(nn.Module):
    def __init__(self, num_classes, backbone='resnet50'):
        super(CarMakeModel, self).__init__()
        
        if backbone == 'resnet18':
            self.base_model = models.resnet18(pretrained=True)
            in_features = 512
        elif backbone == 'resnet50':
            self.base_model = models.resnet50(pretrained=True)
            in_features = 2048
        else:
            raise ValueError("Backbone must be 'resnet18' or 'resnet50'")
        
        # Remplacer la dernière couche
        self.base_model.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        return self.base_model(x)

# Initialiser le modèle
num_make_classes = len(full_dataset.make_to_idx)
make_model = CarMakeModel(num_make_classes, backbone='resnet50')
make_model = make_model.to(device)

print(f"Modèle initialisé avec {num_make_classes} classes")

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
        
        # Chaque epoch a une phase d'entraînement et de validation
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader
            
            running_loss = 0.0
            running_corrects = 0
            
            # Itérer sur les données
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Remettre à zéro les gradients
                optimizer.zero_grad()
                
                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    # Backward + optimizer seulement en phase train
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                # Statistiques
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            if phase == 'train':
                scheduler.step()
            
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            
            if phase == 'train':
                train_losses.append(epoch_loss)
                train_accs.append(epoch_acc.item())
            else:
                val_losses.append(epoch_loss)
                val_accs.append(epoch_acc.item())
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # Sauvegarder le meilleur modèle
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        
        print()
    
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    
    # Charger les poids du meilleur modèle
    model.load_state_dict(best_model_wts)
    
    return model, train_losses, val_losses, train_accs, val_accs

# Définir la loss et l'optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(make_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Entraîner le modèle
make_model, train_loss, val_loss, train_acc, val_acc = train_model(
    make_model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=15
)

# Sauvegarder le modèle de marque
torch.save(make_model.state_dict(), 'car_make_model.pth')
print("Modèle de marque sauvegardé!")

In [ ]:
class CarModelDataset(Dataset):
    def __init__(self, base_path="data/image", transform=None, make_model=None):
        self.base_path = base_path
        self.transform = transform
        self.make_model = make_model  # Modèle pré-entraîné pour les features
        self.data = []
        self.model_to_idx = {}
        self.idx_to_model = {}
        
        # Parcourir la structure et créer le mapping
        if os.path.exists(base_path):
            model_counter = 0
            
            for make_id in os.listdir(base_path):
                make_path = os.path.join(base_path, make_id)
                if os.path.isdir(make_path):
                    for model_id in os.listdir(make_path):
                        model_path = os.path.join(make_path, model_id)
                        if os.path.isdir(model_path):
                            # Créer un identifiant unique pour le modèle
                            unique_model_id = f"{make_id}_{model_id}"
                            if unique_model_id not in self.model_to_idx:
                                self.model_to_idx[unique_model_id] = model_counter
                                self.idx_to_model[model_counter] = unique_model_id
                                model_counter += 1
                            
                            model_idx = self.model_to_idx[unique_model_id]
                            
                            for year in os.listdir(model_path):
                                year_path = os.path.join(model_path, year)
                                if os.path.isdir(year_path):
                                    for img_file in os.listdir(year_path):
                                        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                            img_path = os.path.join(year_path, img_file)
                                            self.data.append((img_path, model_idx))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, model_idx = self.data[idx]
        
        # Charger l'image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, model_idx

# Créer le dataset pour les modèles
model_dataset = CarModelDataset(transform=val_transform, make_model=make_model)

# Split train/val
train_size_model = int(0.8 * len(model_dataset))
val_size_model = len(model_dataset) - train_size_model
train_dataset_model, val_dataset_model = torch.utils.data.random_split(
    model_dataset, [train_size_model, val_size_model]
)

# DataLoaders pour les modèles
train_loader_model = DataLoader(train_dataset_model, batch_size=16, shuffle=True, num_workers=4)
val_loader_model = DataLoader(val_dataset_model, batch_size=16, shuffle=False, num_workers=4)

print(f"Train samples (model): {len(train_dataset_model)}")
print(f"Validation samples (model): {len(val_dataset_model)}")
print(f"Number of model classes: {len(model_dataset.model_to_idx)}")

In [ ]:
class CarModelModel(nn.Module):
    def __init__(self, num_model_classes, make_model):
        super(CarModelModel, self).__init__()
        
        # Utiliser le modèle de marque comme extracteur de features
        self.make_model = make_model
        
        # Geler les poids du modèle de marque
        for param in self.make_model.parameters():
            param.requires_grad = False
        
        # Ajouter de nouvelles couches pour la classification des modèles
        if hasattr(make_model.base_model, 'fc'):
            in_features = make_model.base_model.fc.in_features
            # Remplacer la dernière couche
            make_model.base_model.fc = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_model_classes)
        )
    
    def forward(self, x):
        # Extraire les features avec le modèle de marque
        features = self.make_model.base_model(x)
        # Classification des modèles
        output = self.classifier(features)
        return output

# Initialiser le modèle de modèle
num_model_classes = len(model_dataset.model_to_idx)
car_model_model = CarModelModel(num_model_classes, make_model)
car_model_model = car_model_model.to(device)

print(f"Modèle de modèle initialisé avec {num_model_classes} classes")

In [ ]:
# Définir la loss et l'optimizer pour le modèle de modèle
criterion_model = nn.CrossEntropyLoss()
optimizer_model = optim.Adam(car_model_model.classifier.parameters(), lr=0.001)
scheduler_model = optim.lr_scheduler.StepLR(optimizer_model, step_size=7, gamma=0.1)

# Entraîner le modèle de modèle
car_model_model, train_loss_model, val_loss_model, train_acc_model, val_acc_model = train_model(
    car_model_model, train_loader_model, val_loader_model, 
    criterion_model, optimizer_model, scheduler_model, num_epochs=10
)

# Sauvegarder le modèle complet
torch.save(car_model_model.state_dict(), 'car_model_complete.pth')
print("Modèle complet sauvegardé!")

In [ ]:
# Exporter le modèle en TorchScript
def export_to_torchscript(model, output_path="car_model_scripted.pt"):
    # Mettre le modèle en mode evaluation
    model.eval()
    
    # Exemple d'input pour le tracing
    example_input = torch.randn(1, 3, 224, 224).to(device)
    
    # Scripted model
    scripted_model = torch.jit.trace(model, example_input)
    
    # Sauvegarder
    scripted_model.save(output_path)
    print(f"Modèle exporté en TorchScript: {output_path}")
    
    return scripted_model

# Exporter le modèle
scripted_model = export_to_torchscript(car_model_model)

# Sauvegarder les mappings pour l'inférence
model_info = {
    'make_to_idx': full_dataset.make_to_idx,
    'idx_to_make': full_dataset.idx_to_make,
    'model_to_idx': model_dataset.model_to_idx,
    'idx_to_model': model_dataset.idx_to_model,
    'make_names': make_names
}

torch.save(model_info, 'model_mappings.pth')
print("Mappings sauvegardés!")

In [ ]:
class CarRecognizer:
    def __init__(self, model_path, mappings_path, device='cpu'):
        self.device = device
        self.model = torch.jit.load(model_path, map_location=device)
        self.model.eval()
        
        # Charger les mappings
        mappings = torch.load(mappings_path, map_location=device)
        self.make_to_idx = mappings['make_to_idx']
        self.idx_to_make = mappings['idx_to_make']
        self.model_to_idx = mappings['model_to_idx']
        self.idx_to_model = mappings['idx_to_model']
        self.make_names = mappings['make_names']
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def predict(self, image_path):
        # Charger et prétraiter l'image
        image = Image.open(image_path).convert('RGB')
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # Prédiction
        with torch.no_grad():
            outputs = self.model(input_tensor)
            _, predicted = torch.max(outputs, 1)
            model_idx = predicted.item()
        
        # Convertir l'index en nom de modèle
        model_id = self.idx_to_model[model_idx]
        make_id, specific_model_id = model_id.split('_')
        
        # Trouver le nom de la marque
        make_name = self.make_names[int(make_id)] if int(make_id) < len(self.make_names) else f"Make_{make_id}"
        
        return {
            'make_id': make_id,
            'make_name': make_name,
            'model_id': specific_model_id,
            'confidence': torch.nn.functional.softmax(outputs, dim=1)[0, model_idx].item()
        }

# Exemple d'utilisation
recognizer = CarRecognizer('car_model_scripted.pt', 'model_mappings.pth', device='cpu')

# Tester avec une image
# result = recognizer.predict('chemin/vers/ton/image.jpg')
# print(f"Marque: {result['make_name']}")
# print(f"Modèle ID: {result['model_id']}")
# print(f"Confiance: {result['confidence']:.4f}")